#### Semantic Search with LLM for Grocery

We are using below things:

Database: ChromaDB (Pre loaded)
Embedding Model: sentence-transformers/all-MiniLM-L6-v2  from hugging face

##### Reading Data From Stored VectorDB

In [ ]:
# let's use langchain, here we have to install native chromadb support for langchain as our db is created using native lib
from langchain_chroma import Chroma

# importing embedding function instead from native let's use langchain-huggingface lib 
from langchain_huggingface import HuggingFaceEmbeddings

let's initialize the embedding function which we used to creat the embeddings for our data

In [ ]:
st_ef = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

let's create the client for chromadb

In [ ]:
stored_db = './chroma_store'
collection_name = "grocery_collection"
db = Chroma(
    embedding_function=st_ef,
    persist_directory=stored_db,
    collection_name=collection_name
)

perfect, let's query

In [ ]:
db.similarity_search("healthy snacks which is low in fat", k=5)

Awesome!!! We used different libs to get the result from chromaDB. Now let's add llm so that we can get final refined answer

##### Adding LLM to get the Final Answer

We are going to use langchain retriever - chain concept

Step 1: Load the llm

In [ ]:
# we are using here pipeline (download into local device) instead endpoint, we could also use endpoint which calls Free APIs
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [ ]:
local_hfp = HuggingFacePipeline.from_model_id(
    model_id="google/gemma-3-270m-it",  # it version is like chat
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=512,
        repetition_penalty=1.05,
        do_sample=False
    ),
    device_map="auto"
)

In [ ]:
from pydantic import BaseModel, Field

class GroceryProduct(BaseModel): 
    category: str = Field(description="Category of the product")
    sub_category: str = Field(description="Sub Category of the product")
    type: str = Field(description="Type of the product")
    description:str =Field(description="One liner product short description. Sould not be more than 200 chars")


class GroceryResults(BaseModel):
    grocery_products: list[GroceryProduct]

In [ ]:
# Here is the catch, we have to use parser as the model which we are using is not having support for direct pydantic validation

from langchain_core.output_parsers import PydanticOutputParser

out_parser = PydanticOutputParser(pydantic_object=GroceryResults)

In [ ]:
# let's load the model with structured output
llm = ChatHuggingFace(llm=local_hfp)

So our llm is ready, 

Step 2: Retriever: let's create the retriever means db as retriever

In [ ]:
retriever = db.as_retriever(search_kwargs={"k":3})

Step 3: Define prompt 

In [ ]:
system_prompt = (
"Your are helpful Grocery Inventory Assistant.\n"
"Your job is to provide best answer to the user from the given 'context' only.\n"
"Do not answer from your own knowledge.\n\n"
"You must output ONLY valid JSON matching the exact schema requested.\n"
"Do NOT include any markdown code blocks, intro text, backticks, or trailing notes.\n\n"
"You must only include you final answer without any comentary or internal process steps"
"Do NOT include any further comentary"
"Format Instructions:\n{format_instructions}"
"Context\n{context}"
)

Step 4: Retriever - Chain : Now we have to create a chain using langchain so that we can implement RAG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [ ]:
# creating prompt strcuture
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")])

In [ ]:
# below is the document QA whcih will fill the result from db into system prompt
qa_chain = create_stuff_documents_chain(llm, prompt)

In [ ]:
# below will be creating RAG pipeline to fetch the data
rag = create_retrieval_chain(retriever, qa_chain)

Perfect, now we can query to RAG

In [ ]:
# need to use same keyword which is been used in system prompt
response = rag.invoke({"input": "healthy snacks which is low in fat",
                      "format_instructions": out_parser.get_format_instructions()})

In [ ]:
# complete response
response

In [ ]:
print(f"User Query: {response['input']}\n")
print(f"LLM Response: {response['answer']}")

In [ ]:
# let's try to get proper object
import json

grocery_products = json.loads(response["answer"].split("```json")[1].replace('\n','').replace("```",''))['grocery_products']
grocery_products

In [ ]:
json.dumps(grocery_products)